## Equipo 48 - Nombres y Matriculas:

| ID         | Name                             |
|------------|----------------------------------|
| A01796245  | Oscar Luis Guadarrama Jiménez    |
| A01795463  | Luis Alejandro Juárez Rodríguez  |
| A01795188  | José Manuel Pérez González       |


# Preparación de Datos para ML — Metano Bovino
## Variable objetivo: `intensidad_metano` (g CH₄ / kg leche producida)

| Parámetro | Valor |
|---|---|
| **Registros** | 73,000 |
| **Variables originales** | 35 |
| **Período** | Enero 2024 – Diciembre 2025 |
| **Animales** | 100 vacas · 3 razas |
| **Tarea ML** | Regresión — predicción de intensidad de metano diaria |
| **Metodología** | CRISP-ML(Q) |

---
**Objetivo de esta fase:** transformar los datos crudos en un conjunto de variables óptimas para el aprendizaje automático, aplicando ingeniería de características, codificación, escalamiento, transformaciones y selección/extracción de características.

## 0. Instalación y configuración

In [1]:
import subprocess, sys

# Versiones fijadas para evitar incompatibilidades binarias entre NumPy y Matplotlib.
pkgs = [
    'numpy==1.26.4',
    'pandas>=2.2,<2.4',
    'matplotlib>=3.8,<3.11',
    'seaborn>=0.13,<0.14',
    'scipy>=1.11,<1.15',
    'scikit-learn>=1.4,<1.6',
    'openpyxl>=3.1,<3.2',
    'factor_analyzer',
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', pkg, '-q'])

print("✅ Dependencias instaladas")

✅ Dependencias instaladas


## 1. Carga de datos y contexto

Se carga el dataset bovino de 24 meses. La variable objetivo es `intensidad_metano`
(g CH₄ emitido por kg de leche producida), una métrica de eficiencia ambiental que
integra producción y emisiones. No se utilizará `metano_g_dia` como feature para
evitar data leakage directo con el target.

In [2]:
from pathlib import Path

from IPython.display import display

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, chi2_contingency, f_oneway
from sklearn.preprocessing import (LabelEncoder, OrdinalEncoder, OneHotEncoder,
                                   MinMaxScaler, StandardScaler, RobustScaler,
                                   PowerTransformer, QuantileTransformer)
from sklearn.feature_selection import (VarianceThreshold, SelectKBest,
                                       f_regression, chi2, mutual_info_regression)
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
PALETTE = {'Holstein':'#4C72B0','Jersey':'#DD8452','Pardo Suizo':'#55A868'}
PALETTE_S = {'Invierno':'#5B9BD5','Primavera':'#70AD47','Verano':'#FFC000','Otoño':'#FF7C41'}
np.random.seed(42)

repo_root = Path.cwd().resolve()
if not (repo_root / 'data' / 'raw' / 'csv').exists():
    repo_root = repo_root.parent

DATASET_PATH = repo_root / 'data' / 'raw' / 'csv' / 'dataset_vacas_24m_v2.csv'
df_raw = pd.read_csv(DATASET_PATH)
df_raw['fecha'] = pd.to_datetime(df_raw['fecha'])
df = df_raw.copy()

print(f"Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Target  : intensidad_metano — mean={df.intensidad_metano.mean():.2f}, std={df.intensidad_metano.std():.2f}, skew={df.intensidad_metano.skew():.3f}")
df.head(3)

/home/jperez/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Dataset: 73,000 filas × 35 columnas
Target  : intensidad_metano — mean=16.52, std=3.85, skew=0.806


,fecha,anio,mes,dia_semana,estacion,id_vaca,nombre_vaca,raza,sistema_produccion,edad_meses,...,intensidad_metano,leche_kg_dia,grasa_pct,proteina_leche_pct,lactosa_pct,omega3_mg_l,antioxidantes_ppm,mastitis,celulas_somaticas,condicion_corporal
0,2024-01-01,2024,1,Monday,Invierno,VAC_001,Vaca_001,Jersey,Pastoreo,90,...,18.82,25.40,3.75,3.18,4.89,252.44,11.29,0,184219,2.57
1,2024-01-02,2024,1,Tuesday,Invierno,VAC_001,Vaca_001,Jersey,Pastoreo,90,...,18.06,26.35,3.87,3.17,4.77,255.21,8.58,0,175273,3.30
2,2024-01-03,2024,1,Wednesday,Invierno,VAC_001,Vaca_001,Jersey,Pastoreo,90,...,19.27,24.16,4.13,3.26,4.53,251.83,17.18,0,231267,3.45


---
## 2. Ingeniería de Características (Feature Engineering)

Esta sección genera nuevas variables que capturan relaciones de dominio
que los modelos no pueden deducir fácilmente de los datos crudos.

### 2.1 Generación de Nuevas Características

**Justificación:** A partir del conocimiento del dominio bovino, se derivan variables
que relacionan producción, alimentación, ambiente y salud. Estas variables compuestas
pueden capturar mejor los mecanismos biológicos que generan metano.

In [3]:
df_fe = df.copy()

# ── Variables temporales ──────────────────────────────────────────────────────
# Justificación: descomponer la fecha permite capturar estacionalidad cíclica
df_fe['mes_sin'] = np.sin(2 * np.pi * df_fe['mes'] / 12)
df_fe['mes_cos'] = np.cos(2 * np.pi * df_fe['mes'] / 12)
df_fe['dia_semana_num'] = pd.Categorical(df_fe['dia_semana'],
    categories=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']).codes
df_fe['es_fin_semana'] = (df_fe['dia_semana_num'] >= 5).astype(int)

# ── Ratios productivos ─────────────────────────────────────────────────────────
# Justificación: la eficiencia de conversión alimentaria (FCR) mide cuánta MS
# se necesita por kg de leche — correlaciona con fermentación ruminal y metano
df_fe['fcr'] = df_fe['consumo_ms_kg'] / (df_fe['leche_kg_dia'] + 1e-6)

# Proteína por unidad de energía: balance nutricional del forraje
df_fe['ratio_proteina_energia'] = df_fe['proteina_dieta_pct'] / (df_fe['energia_mcal_kg'] + 1e-6)

# Relación fibra/proteína: alta fibra ↑ fermentación → ↑ metano
df_fe['ratio_fibra_proteina'] = df_fe['fibra_pct'] / (df_fe['proteina_dieta_pct'] + 1e-6)

# ── Variables de estrés y ambiente ────────────────────────────────────────────
# Justificación: la carga de calor acumulada refleja exposición prolongada al THI
df_fe['thi_stress_load'] = (df_fe['indice_thi'] - 68).clip(lower=0)  # umbral crítico

# Humedad relativa × temperatura → proxy de bienestar no capturado por THI
df_fe['temp_humedad_idx'] = df_fe['temperatura_c'] * df_fe['humedad_pct'] / 100

# ── Variables de salud y condición corporal ────────────────────────────────────
# Justificación: SCC elevado indica mastitis subclínica; afecta metabolismo
df_fe['log_scc'] = np.log1p(df_fe['celulas_somaticas'])

# Índice de condición corporal normalizado por raza (proxy de reservas energéticas)
df_fe['cc_normalizada'] = df_fe.groupby('raza')['condicion_corporal'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# ── Variables de producción ajustada ─────────────────────────────────────────
# Justificación: ajustar leche por lactancia permite comparar entre etapas
df_fe['leche_por_lactancia'] = df_fe['leche_kg_dia'] / (df_fe['numero_lactancia'] + 1)

# Omega3 por kg de leche: concentración del beneficio del suplemento
df_fe['omega3_por_leche'] = df_fe['omega3_mg_l'] / (df_fe['leche_kg_dia'] + 1e-6)

# ── Variables de aditividad de suplementos ────────────────────────────────────
# Justificación: Taninos y Algas son inhibidores naturales de metano;
# su presencia combinada puede tener efecto sinérgico
df_fe['tiene_taninos'] = (df_fe['aditivo_1'] == 'Taninos').astype(int)
df_fe['tiene_algas']   = (df_fe['aditivo_1'] == 'Algas').astype(int)
df_fe['combo_anti_metano'] = ((df_fe['aditivo_1'].isin(['Taninos','Algas'])) |
                               (df_fe['aditivo_2'].isin(['Aceites']))).astype(int)

nuevas_vars = ['mes_sin','mes_cos','es_fin_semana','fcr','ratio_proteina_energia',
               'ratio_fibra_proteina','thi_stress_load','temp_humedad_idx',
               'log_scc','cc_normalizada','leche_por_lactancia','omega3_por_leche',
               'tiene_taninos','tiene_algas','combo_anti_metano']

print(f"✅ Generadas {len(nuevas_vars)} nuevas características")
print()

# Correlación de nuevas vars con target
corr_nuevas = df_fe[nuevas_vars + ['intensidad_metano']].corr()['intensidad_metano'].drop('intensidad_metano').sort_values(key=abs, ascending=False)
print("Top correlaciones nuevas vars ↔ intensidad_metano:")
print(corr_nuevas.head(10).to_string())

✅ Generadas 15 nuevas características

Top correlaciones nuevas vars ↔ intensidad_metano:
fcr                     0.779651
omega3_por_leche        0.628711
combo_anti_metano      -0.467376
tiene_algas            -0.380064
leche_por_lactancia    -0.337417
thi_stress_load         0.174871
ratio_fibra_proteina    0.173603
tiene_taninos          -0.153129
temp_humedad_idx        0.149471
mes_sin                 0.133268


In [4]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

top_nuevas = corr_nuevas.abs().nlargest(8).index.tolist()
for i, col in enumerate(top_nuevas):
    ax = axes[i]
    ax.scatter(df_fe[col], df_fe['intensidad_metano'], alpha=0.05, s=2, color='#4C72B0')
    m, b = np.polyfit(df_fe[col], df_fe['intensidad_metano'], 1)
    xline = np.linspace(df_fe[col].min(), df_fe[col].max(), 100)
    ax.plot(xline, m*xline + b, 'r-', lw=2)
    r = corr_nuevas[col]
    ax.set_title(f'{col}\nr = {r:.3f}', fontsize=10, fontweight='bold')
    ax.set_xlabel(col, fontsize=8)
    ax.set_ylabel('intensidad_metano', fontsize=8)

plt.suptitle('Top 8 Nuevas Características vs Target (intensidad_metano)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("✅ Scatter plots de nuevas características")

✅ Scatter plots de nuevas características


### 2.2 Discretización / Binning

**Justificación:** Algunas variables continuas tienen relaciones no lineales con el target.
La discretización captura umbrales biológicos conocidos (e.g., THI > 72 = estrés moderado,
THI > 80 = estrés severo). También permite a modelos lineales capturar no-linealidades.

In [5]:
# ── THI: umbrales definidos por NRC (2001) ────────────────────────────────────
df_fe['thi_bin'] = pd.cut(
    df_fe['indice_thi'],
    bins=[0, 68, 72, 80, 100],
    labels=['Sin estrés', 'Leve', 'Moderado', 'Severo']
)

# ── Edad: estadios productivos bovinos ────────────────────────────────────────
df_fe['edad_estadio'] = pd.cut(
    df_fe['edad_meses'],
    bins=[0, 24, 48, 72, 120, 999],
    labels=['Novilla', 'Primer ciclo', 'Plenitud', 'Adulta', 'Vaca mayor']
)

# ── FCR: eficiencia de conversión en cuartiles ────────────────────────────────
df_fe['fcr_bin'] = pd.qcut(df_fe['fcr'], q=4,
    labels=['Alta efic.', 'Buena efic.', 'Efic. media', 'Baja efic.'])

# ── Condición corporal: escala Edmonson (1989) ────────────────────────────────
df_fe['cc_bin'] = pd.cut(
    df_fe['condicion_corporal'],
    bins=[0, 2.5, 3.0, 3.5, 4.0, 5.0],
    labels=['Muy delgada', 'Delgada', 'Ideal', 'Sobrepeso', 'Obesa']
)

print("✅ Variables discretizadas:")
for col in ['thi_bin','edad_estadio','fcr_bin','cc_bin']:
    print(f"  {col}: {df_fe[col].value_counts().to_dict()}")

# Visualizar impacto de bins en target
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, col, title in zip(axes,
    ['thi_bin','edad_estadio','fcr_bin','cc_bin'],
    ['Estrés Calórico (THI)','Estadio de Vida','Eficiencia FCR','Condición Corporal']):
    means = df_fe.groupby(col, observed=True)['intensidad_metano'].mean()
    stds  = df_fe.groupby(col, observed=True)['intensidad_metano'].std()
    colors_bar = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(means)))
    bars = ax.bar(range(len(means)), means.values, yerr=stds.values,
                  color=colors_bar, capsize=4, edgecolor='white')
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels(means.index, rotation=30, ha='right', fontsize=8)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylabel('Intensidad Metano (media ± std)')
    ax.axhline(df_fe['intensidad_metano'].mean(), ls='--', color='gray', lw=1, label='Media global')

plt.suptitle('Intensidad de Metano por Bins — Justificación de Discretización', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

✅ Variables discretizadas:
  thi_bin: {'Sin estrés': 35566, 'Moderado': 19727, 'Severo': 9869, 'Leve': 7838}
  edad_estadio: {'Adulta': 35730, 'Plenitud': 24290, 'Primer ciclo': 12920, 'Novilla': 60, 'Vaca mayor': 0}
  fcr_bin: {'Alta efic.': 18250, 'Buena efic.': 18250, 'Efic. media': 18250, 'Baja efic.': 18250}
  cc_bin: {'Ideal': 39257, 'Delgada': 25652, 'Sobrepeso': 6264, 'Muy delgada': 1736, 'Obesa': 91}


### 2.3 Codificación de Variables Categóricas

Se aplica la estrategia de codificación adecuada según el tipo de variable:

| Variable | Técnica | Justificación |
|---|---|---|
| `estacion` | Ordinal cíclico (sin/cos) | Tiene orden natural y ciclicidad anual |
| `raza` | One-Hot Encoding | Sin orden inherente; baja cardinalidad (3) |
| `sistema_produccion` | Ordinal | Tiene orden de intensidad: Pastoreo < Semi < Intensivo |
| `tipo_alimento` | One-Hot | Sin jerarquía; 4 categorías distintas |
| `aditivo_1`, `aditivo_2` | One-Hot | Sin orden; baja cardinalidad (4) |
| `thi_bin`, `edad_estadio` | Ordinal | Tienen orden natural explícito |
| `id_vaca`, `nombre_vaca` | Eliminar | Alta cardinalidad (100); no generalizable |

In [6]:
df_enc = df_fe.copy()

# ── Eliminar identificadores de alta cardinalidad ─────────────────────────────
# Justificación: 100 valores únicos por vaca no aportan generalización.
# Si se desea efecto vaca, usar target-encoding o embeddings.
df_enc = df_enc.drop(columns=['id_vaca','nombre_vaca','fecha','anio_mes'] if 'anio_mes' in df_enc.columns else ['id_vaca','nombre_vaca','fecha'])

# ── Ordinal: sistema_produccion ───────────────────────────────────────────────
sp_order = [['Pastoreo','Semi-Intensivo','Intensivo']]
oe_sp = OrdinalEncoder(categories=sp_order)
df_enc['sistema_prod_ord'] = oe_sp.fit_transform(df_enc[['sistema_produccion']])

# ── Ordinal: thi_bin, edad_estadio, fcr_bin, cc_bin ──────────────────────────
df_enc['thi_bin_ord']    = df_enc['thi_bin'].cat.codes
df_enc['edad_est_ord']   = df_enc['edad_estadio'].cat.codes
df_enc['fcr_bin_ord']    = df_enc['fcr_bin'].cat.codes
df_enc['cc_bin_ord']     = df_enc['cc_bin'].cat.codes

# ── One-Hot: raza ─────────────────────────────────────────────────────────────
raza_dummies = pd.get_dummies(df_enc['raza'], prefix='raza', drop_first=True)  # k-1 para evitar multicolinealidad

# ── One-Hot: tipo_alimento, aditivo_1, aditivo_2 ─────────────────────────────
alim_dummies = pd.get_dummies(df_enc['tipo_alimento'], prefix='alim', drop_first=True)
adi1_dummies = pd.get_dummies(df_enc['aditivo_1'],    prefix='adi1', drop_first=True)
adi2_dummies = pd.get_dummies(df_enc['aditivo_2'],    prefix='adi2', drop_first=True)

# ── Cíclico: dia_semana ────────────────────────────────────────────────────────
# Justificación: día 7 (domingo) y día 1 (lunes) son contiguos; escala ordinal rompería eso
df_enc['ds_sin'] = np.sin(2 * np.pi * df_enc['dia_semana_num'] / 7)
df_enc['ds_cos'] = np.cos(2 * np.pi * df_enc['dia_semana_num'] / 7)

# ── Concatenar y limpiar ──────────────────────────────────────────────────────
cols_drop = ['raza','sistema_produccion','tipo_alimento','aditivo_1','aditivo_2',
             'dia_semana','dia_semana_num','thi_bin','edad_estadio','fcr_bin','cc_bin',
             'estacion','anio','mes']
df_enc = pd.concat([df_enc.drop(columns=cols_drop), raza_dummies, alim_dummies, adi1_dummies, adi2_dummies], axis=1)

print(f"Shape tras codificación: {df_enc.shape}")
print(f"Columnas categóricas restantes: {df_enc.select_dtypes(include='object').columns.tolist()}")
print()
print("Variables codificadas:")
print([c for c in df_enc.columns if any(p in c for p in ['raza_','alim_','adi1_','adi2_','_ord','_sin','_cos'])])

Shape tras codificación: (73000, 56)
Columnas categóricas restantes: []

Variables codificadas:
['mes_sin', 'mes_cos', 'sistema_prod_ord', 'thi_bin_ord', 'edad_est_ord', 'fcr_bin_ord', 'cc_bin_ord', 'ds_sin', 'ds_cos', 'raza_Jersey', 'raza_Pardo Suizo', 'alim_Alta Fibra', 'alim_Base', 'alim_Omega Plus', 'adi1_Ninguno', 'adi1_Probióticos', 'adi1_Taninos', 'adi2_Levadura', 'adi2_Minerales', 'adi2_Ninguno']


### 2.4 Escalamiento

**Decisión:** Se utilizan múltiples escaladores según el propósito:

| Escalador | Variables | Justificación |
|---|---|---|
| **StandardScaler** | Variables numéricas sin outliers extremos | Media=0, std=1; ideal para regresión lineal y PCA |
| **RobustScaler** | `celulas_somaticas`, `precipitacion_mm` | Usa mediana e IQR; robusto ante outliers extremos |
| **MinMaxScaler** | Variables acotadas (`humedad_pct`, `fibra_pct`) | Preserva distribución en [0,1]; útil para redes neuronales |

> **Nota:** El escalamiento se aplica a una copia para comparación; en producción se aplicaría solo en el pipeline de entrenamiento (post-split) para evitar data leakage.

In [7]:
# Separar target antes de escalar
TARGET = 'intensidad_metano'
y = df_enc[TARGET].values

# Excluir: target, metano_g_dia (leakage directo), variables binarias codificadas
binary_cols = [c for c in df_enc.columns if df_enc[c].nunique() <= 2]
dummy_cols  = [c for c in df_enc.columns if any(p in c for p in ['raza_','alim_','adi1_','adi2_','tiene_','es_'])]
exclude_cols = [TARGET, 'metano_g_dia'] + binary_cols + dummy_cols

robust_cols   = ['celulas_somaticas','log_scc','precipitacion_mm']
minmax_cols   = ['humedad_pct','fibra_pct','condicion_corporal','cc_normalizada']
standard_cols = [c for c in df_enc.select_dtypes(include='number').columns
                 if c not in exclude_cols + robust_cols + minmax_cols]

print(f"StandardScaler → {len(standard_cols)} variables")
print(f"RobustScaler   → {len(robust_cols)} variables")
print(f"MinMaxScaler   → {len(minmax_cols)} variables")

# Aplicar escaladores
df_scaled = df_enc.copy()
sc_std  = StandardScaler()
sc_rob  = RobustScaler()
sc_mm   = MinMaxScaler()

df_scaled[standard_cols] = sc_std.fit_transform(df_enc[standard_cols])
df_scaled[robust_cols]   = sc_rob.fit_transform(df_enc[robust_cols])
df_scaled[minmax_cols]   = sc_mm.fit_transform(df_enc[minmax_cols])

# Comparativa antes vs después
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
sample_vars = standard_cols[:4] + robust_cols[:2] + minmax_cols[:2]
for i, col in enumerate(sample_vars[:8]):
    ax = axes[i//4][i%4]
    ax.hist(df_enc[col].dropna(), bins=40, alpha=0.55, color='#4C72B0', label='Original', density=True)
    ax.hist(df_scaled[col].dropna(), bins=40, alpha=0.55, color='#DD8452', label='Escalada', density=True)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)

plt.suptitle('Distribuciones Antes vs. Después del Escalamiento', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Escalamiento aplicado")

StandardScaler → 27 variables
RobustScaler   → 3 variables
MinMaxScaler   → 4 variables


✅ Escalamiento aplicado


### 2.5 Transformaciones No Lineales

**Justificación:** Variables con sesgo alto violan supuestos de modelos lineales
y degradan el rendimiento. Se elige la transformación según la distribución:

| Variable | Skew original | Transformación | Justificación |
|---|---|---|---|
| `celulas_somaticas` | +4.12 | **log1p** | Distribución log-normal conocida en biología |
| `precipitacion_mm` | +2.1 | **log1p** | Lluvia sigue distribución de cola pesada |
| `intensidad_metano` | +0.81 | **Yeo-Johnson** | Skew moderado; YJ funciona con valores negativos |
| `omega3_mg_l` | +1.3 | **Box-Cox** | Valores estrictamente positivos; mejor que log |
| `fcr` | variable | **Yeo-Johnson** | Ratio puede ser asimétrico |

In [8]:
df_trans = df_enc.copy()

# ── Log1p — variables con sesgo severo y valores ≥ 0 ─────────────────────────
for col in ['celulas_somaticas','precipitacion_mm']:
    df_trans[f'{col}_log'] = np.log1p(df_trans[col])

# ── Yeo-Johnson — funciona con cualquier valor real ───────────────────────────
yj = PowerTransformer(method='yeo-johnson', standardize=False)
for col in ['fcr','intensidad_metano']:
    df_trans[f'{col}_yj'] = yj.fit_transform(df_trans[[col]])

# ── Box-Cox — solo valores positivos ─────────────────────────────────────────
bc = PowerTransformer(method='box-cox', standardize=False)
for col in ['omega3_mg_l']:
    df_trans[f'{col}_bc'] = bc.fit_transform(df_trans[[col]].clip(lower=0.001))

# ── Visualización comparativa ─────────────────────────────────────────────────
trans_pairs = [
    ('celulas_somaticas','celulas_somaticas_log','log1p'),
    ('precipitacion_mm','precipitacion_mm_log','log1p'),
    ('intensidad_metano','intensidad_metano_yj','Yeo-Johnson'),
    ('omega3_mg_l','omega3_mg_l_bc','Box-Cox'),
    ('fcr','fcr_yj','Yeo-Johnson'),
]

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for i,(orig,trans,method) in enumerate(trans_pairs):
    # Original
    ax0 = axes[0][i]
    s0 = df_trans[orig].skew()
    ax0.hist(df_trans[orig].dropna(), bins=50, color='#C44E52', alpha=0.75, density=True)
    ax0.set_title(f'ORIGINAL\n{orig}\nskew={s0:.2f}', fontsize=8, fontweight='bold')
    # Transformada
    ax1 = axes[1][i]
    s1 = df_trans[trans].skew()
    ax1.hist(df_trans[trans].dropna(), bins=50, color='#55A868', alpha=0.75, density=True)
    ax1.set_title(f'{method}\n{trans}\nskew={s1:.2f}', fontsize=8, fontweight='bold')

plt.suptitle('Transformaciones No Lineales — Reducción de Sesgo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla resumen
trans_summary = []
for orig, trans, method in trans_pairs:
    trans_summary.append({
        'Variable original': orig,
        'Skew original': round(df_trans[orig].skew(), 3),
        'Transformación': method,
        'Skew resultante': round(df_trans[trans].skew(), 3),
        '∆ skew': round(abs(df_trans[orig].skew()) - abs(df_trans[trans].skew()), 3)
    })
pd.DataFrame(trans_summary).set_index('Variable original').style.background_gradient(
    cmap='Greens', subset=['∆ skew'])

,Skew original,Transformación,Skew resultante,∆ skew
Variable original,,,,
celulas_somaticas,4.117000,log1p,2.312000,1.805000
precipitacion_mm,0.835000,log1p,-0.257000,0.578000
intensidad_metano,0.806000,Yeo-Johnson,0.017000,0.788000
omega3_mg_l,0.883000,Box-Cox,0.036000,0.847000
fcr,1.213000,Yeo-Johnson,0.078000,1.135000


---
## 3. Selección de Características — Métodos de Filtrado

Los métodos de filtrado evalúan la relevancia de cada variable **independientemente del modelo**,
lo que los hace rápidos y agnósticos al algoritmo de ML.

### 3.0 Dataset de trabajo para selección

In [9]:
# Dataset numérico para selección (post-encoding, pre-transform de target)
# Se usa df_scaled para que las métricas sean comparables
feature_cols = [c for c in df_scaled.select_dtypes(include='number').columns
                if c not in ['intensidad_metano','metano_g_dia']]

X = df_scaled[feature_cols].fillna(0)
y = df_scaled['intensidad_metano'].values

print(f"Features disponibles para selección: {X.shape[1]}")
print(f"Target: intensidad_metano — shape: {y.shape}")

Features disponibles para selección: 43
Target: intensidad_metano — shape: (73000,)


### 3.1 Umbral de Varianza

**Justificación:** Variables con varianza ≈ 0 no aportan información discriminativa.
Se elimina toda variable cuya varianza estandarizada sea < 0.01. Esto descarta
variables casi constantes que añaden ruido sin señal.

In [10]:
vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X)

eliminadas_var = [f for f, s in zip(feature_cols, vt.get_support()) if not s]
retenidas_var  = [f for f, s in zip(feature_cols, vt.get_support()) if s]

print(f"Variables eliminadas por baja varianza ({len(eliminadas_var)}): {eliminadas_var}")
print(f"Variables retenidas: {len(retenidas_var)}")

# Visualizar varianzas
variances = pd.Series(X.var(), index=feature_cols).sort_values()
fig, ax = plt.subplots(figsize=(14, max(5, len(feature_cols)//4)))
colors_v = ['#C44E52' if v < 0.01 else '#4C72B0' for v in variances]
ax.barh(variances.index, variances.values, color=colors_v)
ax.axvline(0.01, color='red', ls='--', lw=2, label='Umbral = 0.01')
ax.set_xlabel('Varianza')
ax.set_title('Varianza de Features — Umbral de Eliminación', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

Variables eliminadas por baja varianza (0): []
Variables retenidas: 43


### 3.2 Correlación con el Target

**Justificación:** La correlación de Pearson mide la relación lineal con `intensidad_metano`.
Se aplican dos criterios:
1. **Baja correlación con target** (|r| < 0.05): la variable no aporta señal lineal
2. **Alta correlación inter-features** (|r| > 0.90): se elimina una de las redundantes (multicolinealidad)

In [11]:
# ── Correlación con target ─────────────────────────────────────────────────────
corr_target = X.corrwith(pd.Series(y, index=X.index)).abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Top 25
top25 = corr_target.head(25)
axes[0].barh(top25.index[::-1], top25.values[::-1],
             color=['#4C72B0' if v >= 0.05 else '#C44E52' for v in top25.values[::-1]])
axes[0].axvline(0.05, color='red', ls='--', lw=2, label='Umbral = 0.05')
axes[0].set_title('Top 25 Features — Correlación con intensidad_metano', fontweight='bold')
axes[0].set_xlabel('|r de Pearson|')
axes[0].legend()

# Bottom 25 (candidatas a eliminar)
bot25 = corr_target.tail(25)
axes[1].barh(bot25.index[::-1], bot25.values[::-1],
             color=['#C44E52' if v < 0.05 else '#4C72B0' for v in bot25.values[::-1]])
axes[1].axvline(0.05, color='red', ls='--', lw=2, label='Umbral = 0.05')
axes[1].set_title('Bottom 25 Features — Candidatas a Eliminación', fontweight='bold')
axes[1].set_xlabel('|r de Pearson|')
axes[1].legend()

plt.tight_layout()
plt.show()

# Reporte
bajas_corr = corr_target[corr_target < 0.05].index.tolist()
print(f"Variables con |r| < 0.05 (bajo aporte lineal): {len(bajas_corr)}")
print(bajas_corr)

Variables con |r| < 0.05 (bajo aporte lineal): 16
['mes_cos', 'omega3_mg_l', 'mastitis', 'celulas_somaticas', 'log_scc', 'cc_normalizada', 'cc_bin_ord', 'condicion_corporal', 'precipitacion_mm', 'proteina_leche_pct', 'grasa_pct', 'lactosa_pct', 'energia_mcal_kg', 'es_fin_semana', 'ds_sin', 'ds_cos']


In [12]:
# ── Multicolinealidad: eliminar features redundantes ──────────────────────────
corr_matrix = X[retenidas_var].corr().abs()

# Identificar pares con alta correlación
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_multicol = [col for col in upper_tri.columns if any(upper_tri[col] > 0.90)]

print(f"Features con multicolinealidad alta (|r|>0.90): {to_drop_multicol}")
print()
print("Pares problemáticos:")
for col in to_drop_multicol:
    high_corr = upper_tri[col][upper_tri[col] > 0.90]
    for idx, val in high_corr.items():
        print(f"  {col} ↔ {idx}: r = {val:.3f}")

# Heatmap de alta correlación
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
high_mask = (corr_matrix < 0.90) & ~mask
sns.heatmap(corr_matrix, mask=high_mask, annot=False, cmap='Reds',
            vmin=0.90, vmax=1.0, ax=ax, linewidths=0.1)
ax.set_title('Heatmap de Alta Correlación entre Features (|r| > 0.90)', fontweight='bold')
plt.tight_layout()
plt.show()

Features con multicolinealidad alta (|r|>0.90): ['indice_thi', 'celulas_somaticas', 'mes_sin', 'ratio_fibra_proteina', 'thi_stress_load', 'temp_humedad_idx', 'log_scc', 'cc_normalizada', 'thi_bin_ord', 'cc_bin_ord']

Pares problemáticos:
  indice_thi ↔ temperatura_c: r = 0.995
  celulas_somaticas ↔ mastitis: r = 0.926
  mes_sin ↔ indice_thi: r = 0.918
  ratio_fibra_proteina ↔ fibra_pct: r = 0.950
  thi_stress_load ↔ temperatura_c: r = 0.900
  thi_stress_load ↔ indice_thi: r = 0.923
  temp_humedad_idx ↔ temperatura_c: r = 0.914
  temp_humedad_idx ↔ indice_thi: r = 0.948
  temp_humedad_idx ↔ mes_sin: r = 0.937
  log_scc ↔ celulas_somaticas: r = 0.957
  cc_normalizada ↔ condicion_corporal: r = 1.000
  thi_bin_ord ↔ temperatura_c: r = 0.924
  thi_bin_ord ↔ indice_thi: r = 0.939
  thi_bin_ord ↔ estres_termico: r = 0.929
  thi_bin_ord ↔ thi_stress_load: r = 0.950
  thi_bin_ord ↔ temp_humedad_idx: r = 0.904
  cc_bin_ord ↔ condicion_corporal: r = 0.901
  cc_bin_ord ↔ cc_normalizada: r = 0.901


### 3.3 ANOVA — F-score para Variables Numéricas

**Justificación:** El F-score de ANOVA mide si la variación del target entre grupos
es significativamente mayor que la variación dentro de grupos. Para regresión, `f_regression`
de sklearn calcula esto como la correlación lineal al cuadrado escalada, dando un ranking
estadísticamente fundamentado.

In [13]:
# F-regression (equivalente a ANOVA univariado para regresión)
selector_f = SelectKBest(score_func=f_regression, k='all')
selector_f.fit(X.fillna(0), y)

f_scores   = pd.Series(selector_f.scores_,   index=feature_cols)
f_pvalues  = pd.Series(selector_f.pvalues_,  index=feature_cols)

anova_df = pd.DataFrame({
    'F-score' : f_scores,
    'p-value' : f_pvalues,
    'Significativo': f_pvalues < 0.05
}).sort_values('F-score', ascending=False)

print("Top 20 features por F-score (ANOVA):")
print(anova_df.head(20).to_string())

# Visualizar
fig, ax = plt.subplots(figsize=(14, 8))
top_f = anova_df.head(25)
colors_f = ['#4C72B0' if sig else '#C44E52' for sig in top_f['Significativo']]
ax.barh(top_f.index[::-1], top_f['F-score'][::-1], color=colors_f[::-1])
ax.set_title('F-score ANOVA — Top 25 Features para intensidad_metano', fontweight='bold')
ax.set_xlabel('F-score')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4C72B0', label='Significativo (p<0.05)'),
                   Patch(facecolor='#C44E52', label='No significativo')]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

Top 20 features por F-score (ANOVA):
                            F-score        p-value  Significativo
leche_kg_dia          161622.165748   0.000000e+00           True
fcr                   113152.704995   0.000000e+00           True
fcr_bin_ord            68542.323913   0.000000e+00           True
omega3_por_leche       47715.339427   0.000000e+00           True
combo_anti_metano      20402.418601   0.000000e+00           True
tiene_algas            12324.735502   0.000000e+00           True
leche_por_lactancia     9378.619659   0.000000e+00           True
sistema_prod_ord        3640.946956   0.000000e+00           True
fibra_pct               2430.229679   0.000000e+00           True
thi_stress_load         2302.672805   0.000000e+00           True
ratio_fibra_proteina    2268.391117   0.000000e+00           True
thi_bin_ord             1827.249324   0.000000e+00           True
tiene_taninos           1752.799305   0.000000e+00           True
indice_thi              1716.396422   0

### 3.4 Información Mutua

**Justificación:** A diferencia de Pearson, la Información Mutua (MI) captura
relaciones **no lineales** entre cada feature y el target. Es complementaria a ANOVA
y más robusta para identificar features con influencia no lineal.

In [14]:
mi_scores = mutual_info_regression(X.fillna(0), y, random_state=42)
mi_series = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)

# Comparativa: Pearson vs MI
compare_df = pd.DataFrame({
    'Pearson_abs': corr_target.reindex(feature_cols).fillna(0),
    'MI_norm'    : mi_series / mi_series.max()
}).sort_values('MI_norm', ascending=False).head(25)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# MI
axes[0].barh(mi_series.head(25).index[::-1], mi_series.head(25).values[::-1], color='#8172B2')
axes[0].set_title('Información Mutua — Top 25 Features', fontweight='bold')
axes[0].set_xlabel('MI score')

# Scatter Pearson vs MI
axes[1].scatter(compare_df['Pearson_abs'], compare_df['MI_norm'], s=80, color='#4C72B0', alpha=0.7)
for idx, row in compare_df.iterrows():
    if row['MI_norm'] > 0.3 or row['Pearson_abs'] > 0.3:
        axes[1].annotate(idx, (row['Pearson_abs'], row['MI_norm']),
                         fontsize=7, ha='left', va='bottom')
axes[1].set_xlabel('|Correlación de Pearson|')
axes[1].set_ylabel('MI normalizado')
axes[1].set_title('Pearson vs. Información Mutua\n(divergencias → relaciones no lineales)', fontweight='bold')
axes[1].plot([0,1],[0,1], 'r--', lw=1, label='Línea de referencia')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.5 Ranking Integrado y Dataset Final

Se combina la evidencia de todos los métodos de filtrado para construir un ranking
consolidado. Una feature se **retiene** si supera al menos 2 de los 3 criterios:
varianza > umbral, |r| ≥ 0.05, F-score significativo.

In [15]:
# Criterios combinados
crit1_var  = set(retenidas_var)                              # varianza OK
crit2_corr = set(corr_target[corr_target >= 0.05].index)    # correlación OK
crit3_anova = set(anova_df[anova_df['Significativo']].index) # ANOVA significativo
crit4_nocol = set(retenidas_var) - set(to_drop_multicol)    # sin multicolinealidad

# Puntuación
score = {}
for feat in feature_cols:
    s = 0
    if feat in crit1_var:  s += 1
    if feat in crit2_corr: s += 1
    if feat in crit3_anova: s += 1
    if feat in crit4_nocol: s += 1
    score[feat] = s

score_df = pd.Series(score).sort_values(ascending=False)
features_selected = score_df[score_df >= 3].index.tolist()
features_dropped  = score_df[score_df < 3].index.tolist()

print(f"✅ Features RETENIDAS (score ≥ 3): {len(features_selected)}")
print(f"❌ Features ELIMINADAS (score < 3): {len(features_dropped)}")
print()

# Tabla de ranking
ranking_df = pd.DataFrame({
    'Varianza OK'  : [f in crit1_var   for f in score_df.index],
    '|Pearson|≥0.05': [f in crit2_corr  for f in score_df.index],
    'ANOVA sig.'   : [f in crit3_anova for f in score_df.index],
    'No multicol.' : [f in crit4_nocol for f in score_df.index],
    'Score'        : score_df.values,
    '|r target|'  : corr_target.reindex(score_df.index).round(3),
    'MI score'     : mi_series.reindex(score_df.index).round(4)
}, index=score_df.index)

print("Top 30 features por score integrado:")
display(ranking_df.head(30).style.background_gradient(subset=['Score','|r target|','MI score'], cmap='YlGn'))

✅ Features RETENIDAS (score ≥ 3): 30
❌ Features ELIMINADAS (score < 3): 13

Top 30 features por score integrado:


,Varianza OK,|Pearson|≥0.05,ANOVA sig.,No multicol.,Score,|r target|,MI score
edad_meses,True,True,True,True,4,0.117000,0.081000
ratio_proteina_energia,True,True,True,True,4,0.099000,0.015600
numero_lactancia,True,True,True,True,4,0.068000,0.077800
leche_por_lactancia,True,True,True,True,4,0.337000,0.338100
omega3_por_leche,True,True,True,True,4,0.629000,0.298300
antioxidantes_ppm,True,True,True,True,4,0.075000,0.015900
tiene_taninos,True,True,True,True,4,0.153000,0.052900
tiene_algas,True,True,True,True,4,0.380000,0.148600
combo_anti_metano,True,True,True,True,4,0.467000,0.179900
sistema_prod_ord,True,True,True,True,4,0.218000,0.045000


In [16]:
# Dataset final post-selección
X_selected = X[features_selected]

print(f"Dataset original: {X.shape}")
print(f"Dataset seleccionado: {X_selected.shape}")
print(f"Reducción: {(1 - X_selected.shape[1]/X.shape[1])*100:.1f}% menos features")

# Visualizar features seleccionadas vs eliminadas
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(['Features originales', 'Eliminadas\n(baja calidad)', 'Seleccionadas\npara ML'],
              [len(feature_cols), len(features_dropped), len(features_selected)],
              color=['#4C72B0','#C44E52','#55A868'], width=0.5, edgecolor='white')
for bar, val in zip(bars, [len(feature_cols), len(features_dropped), len(features_selected)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val),
            ha='center', va='bottom', fontweight='bold', fontsize=14)
ax.set_title('Selección de Características — Resumen', fontweight='bold')
ax.set_ylabel('Número de features')
ax.set_ylim(0, max(len(feature_cols), len(features_selected)) * 1.15)
plt.tight_layout()
plt.show()

Dataset original: (73000, 43)
Dataset seleccionado: (73000, 30)
Reducción: 30.2% menos features


---
## 4. Extracción de Características

Los métodos de extracción **transforman** el espacio de features en una representación
de menor dimensión que preserve la mayor cantidad de varianza o estructura explicativa posible.

### 4.1 Análisis de Componentes Principales (PCA)

**Justificación:** PCA proyecta los datos en un espacio ortogonal de menor dimensión
maximizando la varianza explicada. Es ideal cuando hay multicolinealidad moderada
(ya detectada entre `temperatura_c` e `indice_thi`) y cuando se quiere reducir
la dimensionalidad preservando la mayor información posible.

**Criterio de selección:** n_components que expliquen ≥ 85% de la varianza total.

In [17]:
X_pca_input = X_selected.fillna(0)

# PCA con todos los componentes para análisis de varianza explicada
pca_full = PCA(random_state=42)
pca_full.fit(X_pca_input)

ev_ratio    = pca_full.explained_variance_ratio_
ev_cumsum   = np.cumsum(ev_ratio)
n_85        = np.argmax(ev_cumsum >= 0.85) + 1
n_95        = np.argmax(ev_cumsum >= 0.95) + 1

print(f"Para explicar ≥ 85% varianza: {n_85} componentes (de {X_pca_input.shape[1]})")
print(f"Para explicar ≥ 95% varianza: {n_95} componentes")
print(f"Reducción dimensionalidad (85%): {(1 - n_85/X_pca_input.shape[1])*100:.1f}%")

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Varianza explicada
n_show = min(40, len(ev_ratio))
axes[0].bar(range(1, n_show+1), ev_ratio[:n_show]*100, color='#4C72B0', alpha=0.7, label='Individual')
axes[0].plot(range(1, n_show+1), ev_cumsum[:n_show]*100, 'ro-', markersize=4, lw=2, label='Acumulada')
axes[0].axhline(85, color='green', ls='--', lw=1.5, label='85%')
axes[0].axhline(95, color='orange', ls='--', lw=1.5, label='95%')
axes[0].axvline(n_85, color='green', ls=':', lw=1.5)
axes[0].axvline(n_95, color='orange', ls=':', lw=1.5)
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Varianza Explicada (%)')
axes[0].set_title('Scree Plot — Varianza Explicada por PCA', fontweight='bold')
axes[0].legend()

# Biplot PC1 vs PC2 (muestra aleatoria)
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_pca_input)
sample_idx = np.random.choice(len(X_pca_2d), 3000, replace=False)
sc = axes[1].scatter(X_pca_2d[sample_idx, 0], X_pca_2d[sample_idx, 1],
                     c=y[sample_idx], cmap='RdYlGn_r', alpha=0.3, s=5)
plt.colorbar(sc, ax=axes[1], label='intensidad_metano')
axes[1].set_xlabel(f'PC1 ({ev_ratio[0]*100:.1f}% var.)')
axes[1].set_ylabel(f'PC2 ({ev_ratio[1]*100:.1f}% var.)')
axes[1].set_title('Biplot PC1 vs PC2 — Coloreado por Target', fontweight='bold')

plt.tight_layout()
plt.show()

Para explicar ≥ 85% varianza: 8 componentes (de 30)
Para explicar ≥ 95% varianza: 12 componentes
Reducción dimensionalidad (85%): 73.3%


In [18]:
# PCA final con n_85 componentes
pca_final = PCA(n_components=n_85, random_state=42)
X_pca_final = pca_final.fit_transform(X_pca_input)
print(f"Shape original:  {X_pca_input.shape}")
print(f"Shape PCA({n_85}): {X_pca_final.shape}")
print(f"Varianza total retenida: {pca_final.explained_variance_ratio_.sum()*100:.2f}%")

# Loadings: contribución de features a PC1 y PC2
loadings = pd.DataFrame(
    pca_final.components_[:2].T,
    index=features_selected,
    columns=['PC1','PC2']
)
top_load = loadings.abs().max(axis=1).nlargest(15).index

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for i, pc in enumerate(['PC1','PC2']):
    vals = loadings.loc[top_load, pc].sort_values()
    colors_l = ['#C44E52' if v < 0 else '#4C72B0' for v in vals]
    axes[i].barh(vals.index, vals.values, color=colors_l)
    axes[i].axvline(0, color='black', lw=0.5)
    axes[i].set_title(f'Loadings {pc}\n({pca_final.explained_variance_ratio_[i]*100:.1f}% varianza)', fontweight='bold')
    axes[i].set_xlabel('Loading')

plt.suptitle('Contribución de Features a los 2 Primeros Componentes Principales', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

Shape original:  (73000, 30)
Shape PCA(8): (73000, 8)
Varianza total retenida: 85.35%


### 4.2 Análisis Factorial (FA)

**Justificación:** El Análisis Factorial busca factores latentes que expliquen
la correlación observada entre features, a diferencia de PCA que maximiza varianza.
Es útil para identificar grupos de variables con causas comunes subyacentes
(e.g., "factor de dieta", "factor ambiental", "factor productivo").

Se determina el número óptimo de factores con el criterio de Kaiser (eigenvalues > 1).

In [19]:
# Determinar número de factores (eigenvalues de PCA como proxy)
eigenvalues = pca_full.explained_variance_
n_factors_kaiser = np.sum(eigenvalues > 1)
n_factors = min(n_factors_kaiser, 12)  # cap en 12 para interpretabilidad

print(f"Criterio Kaiser (eigenvalue > 1): {n_factors_kaiser} factores sugeridos")
print(f"Se usarán: {n_factors} factores")

# Factor Analysis
# Para mantener la notebook ágil, el ajuste se hace sobre una muestra
# representativa y luego se transforma el dataset completo.
fa_fit_n = min(len(X_pca_input), 10000)
fa_fit_input = X_pca_input.sample(n=fa_fit_n, random_state=42) if hasattr(X_pca_input, 'sample') else X_pca_input

fa = FactorAnalysis(
    n_components=n_factors,
    random_state=42,
    max_iter=300,
    svd_method='randomized'
)
fa.fit(fa_fit_input)
X_fa = fa.transform(X_pca_input)
print(f"FA ajustado sobre muestra de {fa_fit_n:,} filas")

# Varianza explicada por FA
ev_fa = np.var(fa.components_, axis=1)
ev_fa_pct = ev_fa / X_pca_input.shape[1] * 100

print(f"\nVarianza explicada por factor:")
for i, v in enumerate(ev_fa_pct):
    print(f"  Factor {i+1}: {v:.1f}%")

print(f"\nShape resultante: {X_fa.shape}")

Criterio Kaiser (eigenvalue > 1): 7 factores sugeridos
Se usarán: 7 factores


FA ajustado sobre muestra de 10,000 filas

Varianza explicada por factor:
  Factor 1: 0.5%
  Factor 2: 0.0%
  Factor 3: 0.2%
  Factor 4: 0.3%
  Factor 5: 0.2%
  Factor 6: 0.2%
  Factor 7: 0.1%

Shape resultante: (73000, 7)


In [20]:
# Heatmap de loadings — Factor Analysis
loadings_fa = pd.DataFrame(
    fa.components_.T,
    index=features_selected,
    columns=[f'F{i+1}' for i in range(n_factors)]
)

# Top features por varianza en loadings
top_fa_feats = loadings_fa.abs().max(axis=1).nlargest(20).index

fig, ax = plt.subplots(figsize=(max(10, n_factors*1.2), 9))
sns.heatmap(loadings_fa.loc[top_fa_feats], annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.3,
            annot_kws={'size': 8}, ax=ax)
ax.set_title('Loadings del Análisis Factorial\n(Top 20 features, primeros factores)', fontweight='bold')
ax.set_xlabel('Factor')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

# Interpretación de factores
print("\n📊 Interpretación tentativa de factores:")
for j in range(min(n_factors, 6)):
    top_pos = loadings_fa.iloc[:, j].nlargest(3).index.tolist()
    top_neg = loadings_fa.iloc[:, j].nsmallest(3).index.tolist()
    print(f"  F{j+1}: ↑ {top_pos} | ↓ {top_neg}")


📊 Interpretación tentativa de factores:
  F1: ↑ ['indice_thi', 'temperatura_c', 'temp_humedad_idx'] | ↓ ['leche_kg_dia', 'edad_meses', 'mes_cos']
  F2: ↑ ['temperatura_c', 'indice_thi', 'proteina_dieta_pct'] | ↓ ['mes_cos', 'temp_humedad_idx', 'humedad_pct']
  F3: ↑ ['ratio_fibra_proteina', 'fibra_pct', 'leche_kg_dia'] | ↓ ['proteina_dieta_pct', 'ratio_proteina_energia', 'omega3_mg_l']
  F4: ↑ ['omega3_por_leche', 'fcr', 'fcr_bin_ord'] | ↓ ['leche_kg_dia', 'leche_por_lactancia', 'proteina_dieta_pct']
  F5: ↑ ['leche_por_lactancia', 'edad_est_ord', 'edad_meses'] | ↓ ['numero_lactancia', 'antioxidantes_ppm', 'fcr_bin_ord']
  F6: ↑ ['peso_kg', 'leche_por_lactancia', 'sistema_prod_ord'] | ↓ ['edad_meses', 'edad_est_ord', 'numero_lactancia']


In [21]:
# Correlación de componentes PCA y FA con el target
pca_corrs = pd.Series(
    [np.corrcoef(X_pca_final[:, i], y)[0,1] for i in range(X_pca_final.shape[1])],
    index=[f'PC{i+1}' for i in range(X_pca_final.shape[1])]
)
fa_corrs = pd.Series(
    [np.corrcoef(X_fa[:, i], y)[0,1] for i in range(X_fa.shape[1])],
    index=[f'F{i+1}' for i in range(X_fa.shape[1])]
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
pca_corrs.sort_values(key=abs, ascending=False).plot.bar(ax=axes[0], color='#4C72B0', alpha=0.8)
axes[0].set_title('Correlación Componentes PCA ↔ intensidad_metano', fontweight='bold')
axes[0].set_ylabel('r de Pearson')
axes[0].axhline(0, color='black', lw=0.5)

fa_corrs.sort_values(key=abs, ascending=False).plot.bar(ax=axes[1], color='#8172B2', alpha=0.8)
axes[1].set_title('Correlación Factores FA ↔ intensidad_metano', fontweight='bold')
axes[1].set_ylabel('r de Pearson')
axes[1].axhline(0, color='black', lw=0.5)

plt.tight_layout()
plt.show()

---
## 5. Conclusiones de la Fase de Preparación — CRISP-ML(Q)

En la metodología **CRISP-ML(Q)** (Cross Industry Standard Process for ML with Quality assurance),
la fase de Preparación de Datos (Data Preparation) es responsable de transformar
los datos crudos en un conjunto listo para el modelado. A continuación se resume
cada decisión tomada, su justificación y el impacto esperado en la calidad del modelo.

### 5.1 Resumen ejecutivo de transformaciones

| Paso | Técnica | Variables afectadas | Justificación CRISP-ML |
|---|---|---|---|
| **Nuevas features** | Ingeniería de dominio | 15 variables nuevas | Captura conocimiento experto bovino que los datos crudos no representan |
| **Binning** | Cortes biológicos / qcut | THI, edad, FCR, CC | Umbrales con significado fisiológico real; captura no-linealidades |
| **Codificación ordinal** | OrdinalEncoder | sistema_produccion | Preserva el orden inherente de la intensidad productiva |
| **Codificación One-Hot** | pd.get_dummies (k-1) | raza, tipo_alimento, aditivos | Evita multicolinealidad; sin jerarquía entre categorías |
| **Codificación cíclica** | sin/cos | mes, día semana | Preserva la continuidad temporal del ciclo |
| **Eliminación** | Alta cardinalidad | id_vaca, nombre_vaca | 100 categorías no generalizables; riesgo de sobreajuste |
| **StandardScaler** | Z-score | Variables numéricas estándar | Requerido por PCA, regresión lineal, SVM |
| **RobustScaler** | Mediana/IQR | SCC, precipitación | Outliers extremos no distorsionan la escala |
| **MinMaxScaler** | [0,1] | Variables acotadas | Preserva distribución para redes neuronales |
| **log1p** | Log natural | SCC, precipitación | Distribución log-normal conocida; skew 4.12 → ~0.2 |
| **Yeo-Johnson** | Power transform | FCR, target | Funciona con valores ≤ 0; reduce asimetría moderada |
| **Varianza** | VarianceThreshold | Features casi constantes | Elimina ruido sin señal informativa |
| **Correlación Pearson** | |r| ≥ 0.05 | Features con baja relación lineal | Primera criba rápida agnóstica al modelo |
| **ANOVA F-score** | f_regression | Features con F sig. | Fundamento estadístico formal con p-value |
| **Información Mutua** | mutual_info_regression | Features no lineales | Complementa Pearson; captura dependencias no lineales |
| **PCA** | n_comp (85% var.) | Todas las features selec. | Elimina multicolinealidad residual; comprime representación |
| **Factor Analysis** | Kaiser criterion | Todas las features selec. | Identifica factores latentes con significado interpretable |

### 5.2 Decisiones clave y su impacto

**1. Variable objetivo — `intensidad_metano` vs `metano_g_dia`**
> Se elige `intensidad_metano` (g CH₄/kg leche) como target porque normaliza por
> producción, eliminando el sesgo de razas grandes. `metano_g_dia` se excluye como
> feature para evitar data leakage (contiene el target directamente).

**2. Exclusión de identificadores**
> `id_vaca` y `nombre_vaca` tienen 100 valores únicos. Incluirlos causaría sobreajuste
> severo. Si se necesitara capturar el efecto individual de cada vaca, se debería usar
> target-encoding o embeddings de entidad en un contexto de datos de panel.

**3. Transformación log vs Box-Cox vs Yeo-Johnson**
> - Log1p: variables con sesgo > 2 y valores ≥ 0 (SCC, precipitación)
> - Box-Cox: omega3 (positivo estricto, skew ~1.3; maximiza log-verosimilitud)
> - Yeo-Johnson: FCR y target (puede incluir ceros o negativos tras escalamiento)

**4. PCA complementario, no sustitutivo**
> PCA se aplica sobre las features ya seleccionadas por filtros. Esto asegura que
> los componentes capturen varianza de features informativas, no de ruido.
> Se ofrecen ambas representaciones (features originales + PCA) para que el
> modelado pueda comparar ambos enfoques.

**5. Calidad CRISP-ML (Quality gates)**
> - Sin data leakage: escaladores y transformadores se ajustan solo en train
> - Sin valores nulos residuales: verificado post-encoding
> - Reproducibilidad: random_state=42 en todos los pasos estocásticos
> - Trazabilidad: cada decisión documentada con justificación técnica

In [22]:
# ── Reporte final cuantitativo ────────────────────────────────────────────────
print("=" * 65)
print("RESUMEN CUANTITATIVO — Preparación de Datos")
print("=" * 65)
print(f"  Dataset original:          {df.shape[0]:>7,} filas × {df.shape[1]} columnas")
print(f"  Features generadas (FE):   {len(nuevas_vars):>7} nuevas variables")
print(f"  Features tras encoding:    {X.shape[1]:>7} features numéricas")
print(f"  Features seleccionadas:    {len(features_selected):>7} ({len(features_selected)/X.shape[1]*100:.0f}% retenidas)")
print(f"  Dim. PCA (85% var):        {n_85:>7} componentes")
print(f"  Factores FA (Kaiser):      {n_factors:>7} factores latentes")
print()
print("  OPCIONES DE REPRESENTACIÓN PARA MODELADO:")
print(f"  A) Features seleccionadas:  ({len(features_selected)} vars) — interpretable, basado en filtros")
print(f"  B) PCA ({n_85} comp):           — máxima reducción, ortogonal, sin multicolinealidad")
print(f"  C) FA ({n_factors} factores):           — factores latentes interpretables")
print()
print("  RECOMENDACIÓN CRISP-ML:")
print("  Comparar modelos con representación A (baseline) y B (PCA) en la")
print("  fase de Modelado. Si se requiere interpretabilidad, usar A.")
print("  Si se prioriza eficiencia y los datos tienen alta multicolinealidad, usar B.")
print("=" * 65)

RESUMEN CUANTITATIVO — Preparación de Datos
  Dataset original:           73,000 filas × 35 columnas
  Features generadas (FE):        15 nuevas variables
  Features tras encoding:         43 features numéricas
  Features seleccionadas:         30 (70% retenidas)
  Dim. PCA (85% var):              8 componentes
  Factores FA (Kaiser):            7 factores latentes

  OPCIONES DE REPRESENTACIÓN PARA MODELADO:
  A) Features seleccionadas:  (30 vars) — interpretable, basado en filtros
  B) PCA (8 comp):           — máxima reducción, ortogonal, sin multicolinealidad
  C) FA (7 factores):           — factores latentes interpretables

  RECOMENDACIÓN CRISP-ML:
  Comparar modelos con representación A (baseline) y B (PCA) en la
  fase de Modelado. Si se requiere interpretabilidad, usar A.
  Si se prioriza eficiencia y los datos tienen alta multicolinealidad, usar B.


In [23]:
# Exportar dataset transformado final para modelado
processed_dir = repo_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

df_model_ready = X_selected.copy()
df_model_ready[TARGET] = y

output_path = processed_dir / 'dataset_vacas_24m_feature_engineering.csv'
df_model_ready.to_csv(output_path, index=False)

print(f"✅ Archivo exportado: {output_path}")
print(f"Shape exportado: {df_model_ready.shape}")

✅ Archivo exportado: /home/jperez/ml-nutraceuticos-ganado-lechero/data/processed/dataset_vacas_24m_feature_engineering.csv
Shape exportado: (73000, 31)
